# Scrapping Allociné

In [5]:
import numpy as np, pandas as pd, requests, re
from bs4 import BeautifulSoup as bs

## Première série sur Allociné : A knight of the Seven Kindoms

### Première page

In [6]:
# Connexion à la page
url = 'https://www.allocine.fr/series/ficheserie-28230/critiques/'

page = requests.get(url)

try:
    page.raise_for_status()
except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

# Création soup
soup = bs(page.text,'html.parser')

### Récupérations des notes

In [7]:
noteSoup = soup.select("span.stareval-note")
note = noteSoup[0].getText().strip()
note

'5,0'

In [8]:
notesAllocine = []

[notesAllocine.append(notes.getText().strip()) for notes in noteSoup]

print(notesAllocine)

['5,0', '5,0', '3,5', '4,5', '4,0', '3,0', '5,0', '4,5', '5,0', '4,0', '3,0', '3,5', '2,0', '1,0', '0,5']


In [9]:
# Remplacer les "," par "."

notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]
notesAllocine

['5.0',
 '5.0',
 '3.5',
 '4.5',
 '4.0',
 '3.0',
 '5.0',
 '4.5',
 '5.0',
 '4.0',
 '3.0',
 '3.5',
 '2.0',
 '1.0',
 '0.5']

### Récupération des avis

In [10]:
avisSoup = soup.select("div.content-txt.review-card-content")
avis = avisSoup[0].getText().strip()
avis

"Un début de saison (Ép. 01) très réussi, qui tranche avec la série principale tout en étant terriblement efficace. On parvient à s'immerger très facilement dans ce monde au réalisme saisissant."

In [11]:
avisAllocine = []

[avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]

print(avisAllocine)

["Un début de saison (Ép. 01) très réussi, qui tranche avec la série principale tout en étant terriblement efficace. On parvient à s'immerger très facilement dans ce monde au réalisme saisissant.", 'J’y ai trouvé du génie ! Pour ceux qui disent «\xa0c’est moue\xa0», revoyez l’épisode 1 de Game of thrones, à l’époque, beaucoup s’était découragé. Moi j’ai adoré !Update suite au visionnage de l’épisode 4, perso j’ai eu la larme à l’œil, c’est magnifique !', 'Quelle drôle d’idée de faire un schema aussi court pour une serie si attendue. On est loin , tout au moins sur le lancement du côté épique, mais il semblerait que l’ on va avoir des personnages plus dejantés, comme des scenes plus terre a terre, comme certains diraient c’est la vie , c’est naturelle, vous comprendrez vite en regardant les premiers scenes, enfinà voir ce que tout ça va donner. On est vraiment dans une bonne serie, bien que le schema court est particulier, bien que comprehensif au vu du peu de scenes d’actions,et par mo

In [12]:
# Nettoyage des caractères spéciaux html du type '\uf449'

avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]

avisAllocine

["Un début de saison (Ép. 01) très réussi, qui tranche avec la série principale tout en étant terriblement efficace. On parvient à s'immerger très facilement dans ce monde au réalisme saisissant.",
 'J’y ai trouvé du génie ! Pour ceux qui disent « c’est moue », revoyez l’épisode 1 de Game of thrones, à l’époque, beaucoup s’était découragé. Moi j’ai adoré !Update suite au visionnage de l’épisode 4, perso j’ai eu la larme à l’œil, c’est magnifique !',
 'Quelle drôle d’idée de faire un schema aussi court pour une serie si attendue. On est loin , tout au moins sur le lancement du côté épique, mais il semblerait que l’ on va avoir des personnages plus dejantés, comme des scenes plus terre a terre, comme certains diraient c’est la vie , c’est naturelle, vous comprendrez vite en regardant les premiers scenes, enfinà voir ce que tout ça va donner. On est vraiment dans une bonne serie, bien que le schema court est particulier, bien que comprehensif au vu du peu de scenes d’actions,et par moment

### Récupérations des dates

In [13]:
dateSoup = soup.select("span.review-card-meta-date.light")
date = dateSoup[0].getText().strip()
date

'Publiée le 19 février 2026'

In [14]:
dateAllocine = []

[dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

print(dateAllocine)

['Publiée le 19 février 2026', 'Publiée le 7 février 2026', 'Publiée le 24 février 2026', 'Publiée le 8 février 2026', 'Publiée le 8 février 2026', 'Publiée le 27 février 2026', 'Publiée le 7 février 2026', 'Publiée le 5 mars 2026', 'Publiée le 10 février 2026', 'Publiée le 3 mars 2026', 'Publiée le 1 mars 2026', 'Publiée le 28 février 2026', 'Publiée le 13 février 2026', 'Publiée le 16 février 2026', 'Publiée le 28 janvier 2026']


In [15]:
# Supprimer 'Publiée le' avant la date

dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]
dateAllocine

['19 février 2026',
 '7 février 2026',
 '24 février 2026',
 '8 février 2026',
 '8 février 2026',
 '27 février 2026',
 '7 février 2026',
 '5 mars 2026',
 '10 février 2026',
 '3 mars 2026',
 '1 mars 2026',
 '28 février 2026',
 '13 février 2026',
 '16 février 2026',
 '28 janvier 2026']

## Récupération des données sur les autres pages

In [16]:
# Création du la soupe sur la barre de navigation
suivantSoup = soup.select("section.section.section-wrap.gd-2-cols.gd-gap-30 > div > nav")
nav = suivantSoup[0]
nav

<nav class="pagination cf"><span class="button button-md button-primary-full button-left button-disabled"><i class="icon icon-left icon-arrow-left"></i><span class="txt">Précédente</span></span><span class="ACrL3NACrlcmllcy9maWNoZXNlcmllLTI4MjMwL2NyaXRpcXVlcy8/cGFnZT0y button button-md button-primary-full button-right"><span class="txt">Suivante</span><i class="icon icon-right icon-arrow-right-a"></i></span><div class="pagination-item-holder"><span class="button button-md item current-item"> 1 </span><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=2">2</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=3">3</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=4">4</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=5">5</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=6">6</a><a class="button button-md item" href="/series/fichese

In [ ]:
# Les liens sont liées aux balises 'a'
page_link = nav.find_all('a')
page_link

[<a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=2">2</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=3">3</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=4">4</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=5">5</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=6">6</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=7">7</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=8">8</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=9">9</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=10">10</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=12">12</a>]

In [18]:
# Compter le nombre de pages

nb_pages = len(page_link) + 1
print(nb_pages)


11


In [19]:
# Vérification des liens URL

relative_url = "https://www.allocine.fr"

for i in range(2, nb_pages+1):  # +1 car Python compte à patir de 0
   print(f"{relative_url}/series/ficheserie-28230/critiques/?page={i}")

https://www.allocine.fr/series/ficheserie-28230/critiques/?page=2
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=3
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=4
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=5
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=6
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=7
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=8
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=9
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=10
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=11


In [20]:
# Récupération des données sur toutes les pages

for i in range(2, nb_pages+1):  # +1 car Python compte à patir de 0
   full_url = f"{relative_url}/series/ficheserie-28230/critiques/?page={i}"
   page = requests.get(full_url)

   try:
    page.raise_for_status()
   except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

   noteSoup = soup.select("span.stareval-note")
   avisSoup = soup.select("div.content-txt.review-card-content")
   dateSoup = soup.select("span.review-card-meta-date.light")

   [notesAllocine.append(notes.getText().strip()) for notes in noteSoup]
   [avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]
   [dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]
avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]
dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]

print(notesAllocine)
print(avisAllocine)
print(dateAllocine)


['5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5', '5.0', '4.0', '3.0', '3.5', '2.0', '1.0', '0.5', '5.0', '5.0', '3.5', '4.5', '4.0', '3.0', '5.0', '4.5'

## Création DataFrame

In [21]:
# Vérification nombre de lignes par liste
print(len(notesAllocine))
print(len(avisAllocine))
print(len(dateAllocine))

colums = ['notesAllocine', 'avis', 'dates']

df_knight = pd.DataFrame(list(zip(notesAllocine, avisAllocine, dateAllocine)), columns = colums)
df_knight["series"] = "A Knight of the Seven Kingdoms"
df_knight

165
165
165


,notesAllocine,avis,dates,series
0,5.0,"Un début de saison (Ép. 01) très réussi, qui t...",19 février 2026,A Knight of the Seven Kingdoms
1,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,7 février 2026,A Knight of the Seven Kingdoms
2,3.5,Quelle drôle d’idée de faire un schema aussi c...,24 février 2026,A Knight of the Seven Kingdoms
3,4.5,Seule le format est décevant. Seulement 6 épis...,8 février 2026,A Knight of the Seven Kingdoms
4,4.0,Je ne suis probablement pas très objectif conc...,8 février 2026,A Knight of the Seven Kingdoms
...,...,...,...,...
160,3.0,Simple mais plutôt efficace. Le format est aty...,1 mars 2026,A Knight of the Seven Kingdoms
161,3.5,Hello World.J’ai un vrai problème avec la dire...,28 février 2026,A Knight of the Seven Kingdoms
162,2.0,Désolé pour ceux qui encensent cette daube mai...,13 février 2026,A Knight of the Seven Kingdoms
163,1.0,Tellement ennuyeux ! Quel intérêt de voir un c...,16 février 2026,A Knight of the Seven Kingdoms


# Série 2 : King & Conqueror

## Première page

In [22]:
# Connexion à la page
url = 'https://www.allocine.fr/series/ficheserie-35604/critiques/'
page = requests.get(url)

try:
    page.raise_for_status()
except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

# Création soups
soup = bs(page.text,'html.parser')
noteSoup = soup.select("span.stareval-note")
avisSoup = soup.select("div.content-txt.review-card-content")
dateSoup = soup.select("span.review-card-meta-date.light")

# Récupération note et premier nettoyage
notesAllocine = []
[notesAllocine.append(notes.getText().strip()) for notes in noteSoup]
notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]

# Récupération avis et premier nettoyage
avisAllocine = []
[avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]
avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]

# Récupération date et premier nettoyage
dateAllocine = []
[dateAllocine.append(date.getText(strip=True)) for date in dateSoup]
dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]

print(notesAllocine)
print(avisAllocine)
print(dateAllocine)


['1.5', '3.0', '0.5', '1.5', '1.5', '2.0', '0.5', '1.5', '1.0', '5.0', '4.0', '1.0', '2.0', '1.0', '2.5']
["J'ai voulu tenter, et j'ai perdu mon temps. La série est cheap plus que de raison, des anachronismes à tout bout de champ pour encore une fois nous faire passer, nous français, pour des guignols qui ne savaient pas se battre (faut-il rappeler que nous sommes la nation à avoir remporté le plus de batailles). Au-delà de l'aspect politique de la série, et si on ne se concentre que sur le visuel, c'est vraiment très très laid, surtout l'étalonnage, tenez-vous prêts à prendre un rendez-vous chez l'ophtalmologiste.", "Une bien belle idée qu'une série sur cette période. Une série historique de plus? C'est en effet cela. C'est plaisant, mais bavard. Le manque de moyen est évident. A défaut de se passionner pour ces discussions de couloirs et de tentes, on s'y intéresse relativement. Doit on parler de vérité historique... C'est tout au plus une incarnation proche d'un certain imaginaire. 

## Autres pages

In [23]:
# Récupération des données sur toutes les pages

suivantSoup = soup.select_one("section.section.section-wrap.gd-2-cols.gd-gap-30 > div > nav")
page_link = suivantSoup.find_all('a')
nb_pages = len(page_link) + 1

for i in range(2, nb_pages+1):  # +1 car Python compte à patir de 0
   full_url = f"{relative_url}/series/ficheserie-35604/critiques/?page={i}"
   page = requests.get(full_url)

   try:
    page.raise_for_status()
   except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

   noteSoup = soup.select("span.stareval-note")
   avisSoup = soup.select("div.content-txt.review-card-content")
   dateSoup = soup.select("span.review-card-meta-date.light")

   [notesAllocine.append(notes.getText().strip()) for notes in noteSoup]
   [avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]
   [dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]
avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]
dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]

print(notesAllocine)
print(avisAllocine)
print(dateAllocine)

['1.5', '3.0', '0.5', '1.5', '1.5', '2.0', '0.5', '1.5', '1.0', '5.0', '4.0', '1.0', '2.0', '1.0', '2.5', '1.5', '3.0', '0.5', '1.5', '1.5', '2.0', '0.5', '1.5', '1.0', '5.0', '4.0', '1.0', '2.0', '1.0', '2.5']
["J'ai voulu tenter, et j'ai perdu mon temps. La série est cheap plus que de raison, des anachronismes à tout bout de champ pour encore une fois nous faire passer, nous français, pour des guignols qui ne savaient pas se battre (faut-il rappeler que nous sommes la nation à avoir remporté le plus de batailles). Au-delà de l'aspect politique de la série, et si on ne se concentre que sur le visuel, c'est vraiment très très laid, surtout l'étalonnage, tenez-vous prêts à prendre un rendez-vous chez l'ophtalmologiste.", "Une bien belle idée qu'une série sur cette période. Une série historique de plus? C'est en effet cela. C'est plaisant, mais bavard. Le manque de moyen est évident. A défaut de se passionner pour ces discussions de couloirs et de tentes, on s'y intéresse relativement. D

# Création Dataframe Global

In [24]:
# Vérification nombre de lignes par liste
print(len(notesAllocine))
print(len(avisAllocine))
print(len(dateAllocine))

columns = ['notesAllocine', 'avis', 'dates']

df_king = pd.DataFrame(list(zip(notesAllocine, avisAllocine, dateAllocine)), columns = columns)
df_king["series"] = "King & Conqueror"
df_king

30
30
30


,notesAllocine,avis,dates,series
0,1.5,"J'ai voulu tenter, et j'ai perdu mon temps. La...",5 mars 2026,King & Conqueror
1,3.0,Une bien belle idée qu'une série sur cette pér...,22 mars 2026,King & Conqueror
2,0.5,"Bon c'est la BBC, donc il fallait s'y attendre...",22 septembre 2025,King & Conqueror
3,1.5,Le rappel à Game of Throne ne tient pas longte...,1 septembre 2025,King & Conqueror
4,1.5,Honnêtement il n’y a pas grand chose de bon à ...,14 février 2026,King & Conqueror
5,2.0,Si vous voulez voir une série basée sur l´acti...,13 septembre 2025,King & Conqueror
6,0.5,Cette production a de nombreux problèmes de ré...,1 septembre 2025,King & Conqueror
7,1.5,"bien réalisé, bien joué, mais l'éternelle mauv...",1 septembre 2025,King & Conqueror
8,1.0,Le point positif les acteurs plutôt bon . Poin...,10 mars 2026,King & Conqueror
9,5.0,Honnêtement j'ai eu du mal à enclenché les 5 p...,5 mars 2026,King & Conqueror


In [ ]:
# Concaténation et conversion des colonnes
df_allocine = pd.concat([df_knight, df_king], ignore_index=True)

# Conversion de la colonne notesAllociné en nombre décimal
df_allocine = df_allocine.astype({'notesAllocine' : 'float'})

# Pour la date : traduction anglais pour permettre la conversion.
mois = {
    'janvier': 'January', 'février': 'February', 'mars': 'March',
    'avril': 'April', 'mai': 'May', 'juin': 'June',
    'juillet': 'July', 'août': 'August', 'septembre': 'September',
    'octobre': 'October', 'novembre': 'November', 'décembre': 'December'
}

df_allocine['dates'] = df_allocine['dates'].replace(mois, regex=True)

# Conversion au format dates
df_allocine['dates'] = pd.to_datetime(df_allocine['dates'], format='%d %B %Y')



In [26]:
# Vérifications

display(df_allocine)
print() # Saut à la ligne
print(df_allocine.info())

,notesAllocine,avis,dates,series
0,5.0,"Un début de saison (Ép. 01) très réussi, qui t...",2026-02-19,A Knight of the Seven Kingdoms
1,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,A Knight of the Seven Kingdoms
2,3.5,Quelle drôle d’idée de faire un schema aussi c...,2026-02-24,A Knight of the Seven Kingdoms
3,4.5,Seule le format est décevant. Seulement 6 épis...,2026-02-08,A Knight of the Seven Kingdoms
4,4.0,Je ne suis probablement pas très objectif conc...,2026-02-08,A Knight of the Seven Kingdoms
...,...,...,...,...
190,4.0,J’ai adoré cette fin. Si l’on aime un spectacl...,2025-08-31,King & Conqueror
191,1.0,Réalisation vieillotte sans budget sans véraci...,2026-03-07,King & Conqueror
192,2.0,Encore une fois le réalisateur écrit sa propre...,2026-02-20,King & Conqueror
193,1.0,"Cette série aurait pu être excellente, avec de...",2026-03-08,King & Conqueror



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   notesAllocine  195 non-null    float64       
 1   avis           195 non-null    object        
 2   dates          195 non-null    datetime64[ns]
 3   series         195 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 6.2+ KB
None


In [ ]:
# Export en csv
df_allocine.to_csv('df_allocine.csv', index=False)